# **Horizon Aware Loss**

In this notebook we are going to explorate the effect of the horizon aware huber loss over the variances of the errors of the predictions.

## **Setup**

These block are dedicated just to set up of the whole notebook.

In [93]:
import matplotlib.pyplot as plt

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn.functional as F

from src.data_handler import *
from src.config_files import *
from src.direct_models import *
from src.mheme import *
from src.metrics import *
from src.plot_handler import *

### Hyperparameters

In [97]:
# Global variables
WINDOW = 48
HORIZON = 24

DATA_PATH = '../data'
DATA_CONFIG_PATH = '../data/data_config.json'

TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config.json'
MSE_TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config_mse.json'
TCN_PATH_SAVE = '../models/'

XGB_PATH_CONFIG_LOAD = '../src/config_files/xgb_config.json'
XGB_PATH_SAVE = '../models/'

ARIMA_PATH_CONFIG_LOAD = '../src/config_files/arima_config.json'

MODELS_PATH_SAVE = '../models/'

### Data

In [106]:
X, data = data_loader(data_path = DATA_PATH, data_config_path= DATA_CONFIG_PATH, dataset_init = 'e')

In [107]:
kwargs = {"name" : "Electricity", "color" : "green", "title" : "Electricity demand over time", "x_axis" : "Time", "y_axis" : "Electricity demand"}
plot_time_series(X, **kwargs)

In [76]:
X_slide, y_slide = sliding_window(X, window=WINDOW, horizon=HORIZON, k = 7)
train, val, test = train_validation_test_split(X_slide, y_slide, shuffle_data= False, shuffle_internal= True, random_state=42)#, shuffle=True, random_state=0) if there is a strong cyclyc trend, no shuffle improves performances

## **Horizon-Aware Huber Loss**

The Horizon-Aware Huber Loss extends the standard Huber loss by introducing horizon-dependent weights that modulate the contribution of each forecast step.\
This formulation enables differential penalization of prediction errors across lead times, thereby encouraging improved accuracy at longer horizons.

Let:

- $B$ be the batch size  
- $H$ the forecast horizon  
- $D$ the number of target variables  
- $ \hat{y}_{b,h,d} $ the prediction at horizon $ h $
- $ y_{b,h,d} $ the corresponding ground-truth value  
- $ w_h \ge 0 $ the horizon-specific weight  
- $ \delta > 0 $ the Huber threshold  

---

### Huber loss (point-wise)

$$
\ell_{\delta}(r) =
\begin{cases}
\frac{1}{2} r^2, & |r| \le \delta \\
\delta \left(|r| - \frac{1}{2}\delta \right), & |r| > \delta
\end{cases}
$$

where the residual is defined as:

$$
r_{b,h,d} = \hat{y}_{b,h,d} - y_{b,h,d}
$$

---

### Horizon-aware Huber loss

$$
\mathcal{L}_{\text{HA-Huber}}
=
\frac{1}{B H D}
\sum_{b=1}^{B}
\sum_{h=1}^{H}
\sum_{d=1}^{D}
w_h \;
\ell_{\delta}
\!\left(
\hat{y}_{b,h,d} - y_{b,h,d}
\right)
$$

---

### Normalized horizon weights (optional)

To avoid unintended gradient scaling, the horizon weights can be normalized as:

$$
\tilde{w}_h
=
\frac{w_h}{\frac{1}{H}\sum_{j=1}^{H} w_j}
$$

and substituted into the loss definition above.

---

### Special cases

- **Uniform weighting**  
  If $ w_h = 1 \;\forall h$ , the horizon-aware Huber loss reduces to the standard Huber loss.

- **Univariate forecasting**  
  If $D = 1$ , the summation over the target dimension is omitted.

---

### Compact formulation

$$
\mathcal{L}
=
\frac{1}{B H D}
\sum_{b,h,d}
w_h \,
\ell_{\delta}
\bigl(
\hat{y}_{b,h,d} - y_{b,h,d}
\bigr)
$$


In [64]:
# residuals
r = np.linspace(-15, 15, 500)
r_t = torch.tensor(r, dtype=torch.float32)

# Huber thresholds to visualize
deltas = [0.5, 1.0, 2.0, 3.0, 10.0]

fig = go.Figure()

for delta in deltas:
    # Huber loss with F.huber_loss requires input and target
    y_hat = r_t.clone()
    y = torch.zeros_like(r_t)
    huber_loss = F.huber_loss(y_hat, y, delta=delta, reduction='none').numpy()
    fig.add_trace(go.Scatter(x=r, y=huber_loss, mode='lines', name=f'Huber δ={delta}'))

fig.update_layout(title="Huber Loss vs Residual", xaxis_title="Residual r = y_pred - y_true", yaxis_title="Huber Loss ℓ_δ(r)", template="plotly_white")

fig.show()

## **Possible Weighting Strategies**

The choice of horizon weights plays a crucial role in the Horizon-Aware Huber Loss, as it determines how prediction errors at different lead times contribute to the overall objective.  
In this work, we consider four different strategies for computing horizon-wise weights.

### Uniform

All forecast horizons are weighted equally.

$$
w_h = 1, \qquad \forall \; h = 1, \dots, H
$$

---

### Soft Linear

The weights increase linearly with the forecast horizon, with a mild emphasis on longer horizons.

$$
w_h = 1 + \frac{h - 1}{H - 1}, \qquad h = 1, \dots, H
$$

This corresponds to a linear interpolation between 1 and 2.

---

### Strong Linear

The weights grow linearly with the forecast horizon, strongly emphasizing long-term predictions.

$$
w_h = h, \qquad h = 1, \dots, H
$$

---

### Exponential

The weights increase exponentially with the forecast horizon, placing progressively higher emphasis on longer lead times.

$$
w_h = \gamma^{\,h - 1}, \qquad h = 1, \dots, H
$$

where $ \gamma > 1 $ controls the rate of exponential growth.


In [65]:
H = 12
h = np.arange(H)

# Weighting strategies
uniform = np.ones(H)
soft_linear = np.linspace(1.0, 5, H)
strong_linear = np.arange(1, H + 1)
gamma = 1.3
exponential = gamma ** h

fig = go.Figure()
fig.add_trace(go.Scatter(x=h, y=uniform, mode="lines", name="Uniform"))
fig.add_trace(go.Scatter(x=h, y=soft_linear, mode="lines", name="Soft Linear (up to 5)"))
fig.add_trace(go.Scatter(x=h, y=strong_linear, mode="lines", name="Strong Linear"))
fig.add_trace(go.Scatter( x=h, y=exponential, mode="lines", name="Exponential (1.3)"))
fig.update_layout(title="Horizon Weighting Strategies", xaxis_title="Forecast Horizon (h)", yaxis_title="Weight $w_h$", legend_title="Strategy", template="plotly_white")

fig.show()


## **Experiment with a TCN**

We investigate how different horizon-weighting strategies in the Horizon-Aware Huber Loss influence the variance of prediction errors across models in an ensemble of Temporal Convolution Networks (TCNs).\
Specifically, we evaluate whether emphasizing longer horizons through various weighting schemes affects error dispersion among ensemble members.

Firstly we visualize what does the normal mse does to the variances of the errors.

### MSE

### Init

Univariate MHEMe ensemble method with TCN models as bases.

In [77]:
mheme_tcn_mse = UMHEMe(HORIZON, WINDOW, TCN, MSE_TCN_PATH_CONFIG_LOAD, skip = 3)

### Fit

With using the MSE Loss

In [78]:
mheme_tcn_mse.fit(train[0], train[1])

Fitting class : <class 'src.direct_models.TCN'>; horizon : 1


Training TCN: 100%|██████████| 100/100 [00:09<00:00, 10.61it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 4


Training TCN: 100%|██████████| 100/100 [00:09<00:00, 10.57it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 7


...

Training TCN: 100%|██████████| 100/100 [00:09<00:00, 10.25it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 19


Training TCN: 100%|██████████| 100/100 [00:09<00:00, 10.52it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 22


Training TCN: 100%|██████████| 100/100 [00:12<00:00,  7.81it/s]


### Prediction

Without tuned weights

In [79]:
pre_preds = mheme_tcn_mse.predict(test[0])
mse(pre_preds, test[1])

tensor(2153.7407)

### Weights computation

On the train set

In [80]:
mheme_tcn_mse.compute_weights(np.concatenate([train[0]], axis = 0), np.concatenate([train[1]], axis = 0))

### Prediction

With tuned weights

In [81]:
after_preds = mheme_tcn_mse.predict(test[0])
mse(after_preds, test[1])

tensor(2070.5117)

In [82]:
mheme_tcn_mse.visualize_variances()
mheme_tcn_mse.visualize_weights(None)
mheme_tcn_mse.visualize_errors(None)

### Horizon aware huber loss

### Init

Univariate MHEMe ensemble method with TCN models as bases.

In [83]:
mheme_tcn_hahl = UMHEMe(HORIZON, WINDOW, TCN, TCN_PATH_CONFIG_LOAD, skip = 3)

### Fit

With the Horizon Aware Huber Loss (exponential decay)

In [84]:
mheme_tcn_hahl.fit(train[0], train[1])

Fitting class : <class 'src.direct_models.TCN'>; horizon : 1


Training TCN: 100%|██████████| 100/100 [00:12<00:00,  7.90it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 4


Training TCN: 100%|██████████| 100/100 [00:10<00:00,  9.23it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 7


...

Training TCN: 100%|██████████| 100/100 [00:12<00:00,  8.25it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 19


Training TCN: 100%|██████████| 100/100 [00:12<00:00,  7.77it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 22


Training TCN: 100%|██████████| 100/100 [00:11<00:00,  8.81it/s]


### Prediction

Without tuned weights

In [87]:
pre_preds = mheme_tcn_hahl.predict(test[0])
mse(pre_preds, test[1])

tensor(2092.0024)

### Weights computation

On the validation set

In [88]:
mheme_tcn_hahl.compute_weights(np.concatenate([train[0]], axis = 0), np.concatenate([train[1]], axis = 0))

### Prediction

With tuned weights

In [89]:
after_preds = mheme_tcn_hahl.predict(test[0])
mse(after_preds, test[1])

tensor(1999.8557)

In [90]:
mheme_tcn_hahl.visualize_variances()
mheme_tcn_hahl.visualize_weights(None)
mheme_tcn_hahl.visualize_errors(None)